<a href="https://colab.research.google.com/github/I-yuki-0424/Decision-Process-order-driven/blob/main/docs/JP-ideas/DPOD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Decision Process Order-Driven

## To best decision system

## Name and Mean

|                  |main | 2nd |
| ---------------- | --- | --- |
| action           | A   |     |
| State            | S   | S_1 |
| target           | T   |     |
| history          | H   |     |
| compression unit | Z   |     |
| candidates       | K   |     |


In [1]:
# just import
import functools
import jax
import jax.numpy as jnp
import flax.linen as nn
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
def get_sinusoidal_positional_encoding(seq_len, d_model):
    """
    sin cos encoder
    """
    position = jnp.arange(seq_len)[:, None]
    div_term = jnp.exp(jnp.arange(0, d_model, 2) * (-jnp.log(10000.0) / d_model))

    pe_sin = jnp.sin(position * div_term)
    pe_cos = jnp.cos(position * div_term)

    pe = jnp.zeros((seq_len, d_model))
    pe = pe.at[:, 0::2].set(pe_sin)
    pe = pe.at[:, 1::2].set(pe_cos)
    return pe

In [3]:
def _y_mdp_action_single(array_to_action, k):
    """
    Original per-sample body of y_mdp_action, unmodified. Operates on a
    single (N, b) slice. Kept separate from y_mdp_action so other batched
    functions can call the per-sample logic directly without a nested vmap.
    """
    S    = array_to_action[:, 0].astype(jnp.int32)
    S_1  = array_to_action[:, 1].astype(jnp.int32)
    A    = array_to_action[:, 2].astype(jnp.int32)
    P    = array_to_action[:, 3]
    R    = array_to_action[:, 4]
    G    = array_to_action[:, 5]

    n = array_to_action.shape[0]                 # static upper bound on state/action ids
    state_action_id = S * n + A                   # unique id per (s, a), valid since A < n

    Q0 = jnp.zeros((n, n), dtype=array_to_action.dtype)  # Q*(s, a) initialized to 0

    def cond_fn(carry):
        it, Q, delta = carry
        return jnp.logical_and(it < n, delta > 1e-6)

    def body_fn(carry):
        it, Q, _ = carry
        topk_next = jax.lax.top_k(Q[S_1], k)[0]         # (num_rows, k)
        soft_max_next = jnp.mean(topk_next, axis=-1)     # (num_rows,)
        transition_value = P * (R + G * soft_max_next)   # (num_rows,)
        Q_flat = jax.ops.segment_sum(transition_value, state_action_id, num_segments=n * n)
        Q_new = Q_flat.reshape(n, n)
        delta = jnp.max(jnp.abs(Q_new - Q))
        return (it + 1, Q_new, delta)

    init_carry = (0, Q0, jnp.asarray(jnp.inf, dtype=array_to_action.dtype))
    _, Q, _ = jax.lax.while_loop(cond_fn, body_fn, init_carry)

    top_values, top_actions = jax.lax.top_k(Q, k)

    return top_actions, top_values


@functools.partial(jax.jit, static_argnames=('k',))
def y_mdp_action(array_to_action, k):
    """
    This is simple **Bellman optimality equation**
    Latex
    Q^*(s, a) = \\sum_{s'} P(s' | s, a) \\left[ R(s, a, s') + \\gamma \\max_{a'} Q^*(s', a') \\right]

    Batch dimension added: array_to_action now has shape (B, N, b) -- B
    batches, N state-action rows per batch, b=6 feature columns (S, S_1,
    A, P, R, G). Per-sample logic is unchanged; jax.vmap maps
    `_y_mdp_action_single` over the leading batch axis. k must be static
    (jax.lax.top_k requires a concrete Python int) -- passing it as an
    ordinary jitted argument, as before, raises a ConcretizationTypeError
    the first time this actually runs; static_argnames fixes that.
    """
    return jax.vmap(_y_mdp_action_single, in_axes=(0, None))(array_to_action, k)

In [4]:
def _make_array_to_action_single(state, next_state, actions, p_coe, r_coe, gamma_coe):
    """
    Original per-sample body. next_state is clamped away from zero before
    division to avoid inf/nan when a next_state entry is exactly (or very
    close to) zero. The ratio-based gradient formula itself is otherwise
    unchanged from the original.
    """
    eps = jnp.asarray(1e-8, dtype=next_state.dtype)
    sign = jnp.where(next_state >= 0, 1.0, -1.0).astype(next_state.dtype)
    safe_next_state = jnp.where(jnp.abs(next_state) < eps, sign * eps, next_state)
    gradient = state / safe_next_state * 100
    P = p_coe * gradient
    R = r_coe * gradient
    G = gamma_coe * gradient

    return (jnp.column_stack([state, next_state, actions, P, R, G]))


#sample
@jax.jit
def make_array_to_action(state, next_state, actions, p_coe, r_coe, gamma_coe):
    """
    S,S' = already,
    a = alredy,
    P = P(%)_coefficient * gradient of S -> S'
    R = R(%)_coefficient * gradient of S -> S'
    Gamma = G(%)_coefficient * gradient of S -> S'
    How to set coefficient? I dont know :/

    Batch dimension added: state, next_state, actions now have shape
    (B, N, b). p_coe, r_coe, gamma_coe are shared coefficients, unbatched.
    next_state is clamped away from zero in the per-sample body to avoid
    inf/nan (see _make_array_to_action_single).
    """
    return jax.vmap(
        _make_array_to_action_single, in_axes=(0, 0, 0, None, None, None)
    )(state, next_state, actions, p_coe, r_coe, gamma_coe)

In [5]:
def _broadcast_to_rows(x, n_rows):
    """
    Broadcast a (1, d) block to (n_rows, d) by repetition; an
    already-(n_rows, d) block passes through unchanged. This assumes
    exactly one 'current' row (state/target) pairs with every action
    candidate -- the same broadcasting Y_attention_state_5_4 does via
    jnp.repeat. If x has some other row count, this does not guess and
    will raise inside jnp.repeat.
    """
    if x.shape[0] == n_rows:
        return x
    return jnp.repeat(x, n_rows, axis=0)


def _add_head_axis(x):
    """
    (length, d) -> (length, 1, d). flax's dot_product_attention requires
    a num_heads axis; this notebook has no multi-head parameter anywhere,
    so a size-1 head is the minimal fix that makes the existing calls
    dimensionally valid without changing what they compute (attention
    with 1 head is mathematically identical to the flat 2-D version the
    code visually intends). Confirmed to raise IndexError without this.
    """
    return x[:, None, :]


def _drop_head_axis(x):
    """(length, 1, d) -> (length, d)."""
    return x[:, 0, :]


def _Y_attention_state_5_1_Bace_single(
    actions, target, state, histry,
    k_s, v_s,
    dropout_rate_s, dropout_enabled_s, rng_s, mask_s, num_l, num_step, k
):
    """Per-sample body. Operates on (N, b) slices."""
    # 1. actions, goal, state concatenate
    q_ags_init = jnp.concatenate([actions, target, state, histry], axis=0)

    def layer_fn(q_carry, _):
        y_attn = nn.dot_product_attention(
            query=_add_head_axis(q_carry),
            key=_add_head_axis(k_s),
            value=_add_head_axis(v_s),
            bias=mask_s,
            dropout_rate=dropout_rate_s,
            deterministic=not dropout_enabled_s,
            dropout_rng=rng_s
        )
        # Residual connection style update
        return q_carry + _drop_head_axis(y_attn), None

    # 2. Layering according to num_l
    y_attention_state, _ = jax.lax.scan(layer_fn, q_ags_init, jnp.arange(num_l))

    n_a, n_t, n_s = actions.shape[0], target.shape[0], state.shape[0]
    mdp_actions, mdp_target, mdp_state, mdp_histry = jnp.split(
        y_attention_state, [n_a, n_a + n_t, n_a + n_t + n_s], axis=0
    )

    # state/target arrive as a single "now"/"goal" row while actions is
    # n_a candidate rows -- broadcast the single row across candidates
    # before stacking columns (mirrors the repeat pattern used in 5_4).
    # NOTE (unresolved, flagged previously): state/mdp_state/actions/
    # mdp_actions are each d-dimensional embeddings, not scalars, so
    # column_stack produces 6*d columns, not the 6 scalar columns
    # (S, S_1, A, P, R, G) y_mdp_action's own indexing expects. This runs
    # without error but is not semantically what y_mdp_action documents.
    state_b = _broadcast_to_rows(state, n_a)
    mdp_state_b = _broadcast_to_rows(mdp_state, n_a)
    target_b = _broadcast_to_rows(target, n_a)

    mdp_array = jnp.column_stack([
        state_b, mdp_state_b, actions, mdp_actions,
        (target_b - mdp_state_b), (target_b - mdp_state_b) * 1 / num_step
    ])
    Y_action = _y_mdp_action_single(mdp_array, k)

    return Y_action


@functools.partial(jax.jit, static_argnames=('num_l', 'k', 'dropout_enabled_s'))
def Y_attention_state_5_1_Bace(
    actions, target, state, histry,
    k_s, v_s,
    dropout_rate_s, dropout_enabled_s, rng_s, mask_s, num_l, num_step, k
):
    """Simple and 5th idea bace

    Batch dimension added: actions, target, state, histry, k_s, v_s,
    mask_s now have shape (B, N, b). num_l, k, and dropout_enabled_s are
    marked static -- num_l is used in jnp.arange() for jax.lax.scan's
    length, k is used in jax.lax.top_k() inside y_mdp_action, and
    dropout_enabled_s is used in a Python `not` boolean check; all three
    require concrete (non-traced) values under jax.jit, confirmed by the
    ConcretizationTypeError this raised without static_argnames.
    dropout_rate_s, rng_s, num_step are shared, unbatched, and do not
    need to be static. The internal call to y_mdp_action uses its
    single-sample form (`_y_mdp_action_single`) to avoid a nested vmap.
    """
    return jax.vmap(
        _Y_attention_state_5_1_Bace_single,
        in_axes=(0, 0, 0, 0, 0, 0, None, None, None, 0, None, None, None)
    )(actions, target, state, histry, k_s, v_s, dropout_rate_s, dropout_enabled_s, rng_s, mask_s, num_l, num_step, k)

In [6]:
def _Y_attention_state_5_2_single(
    actions, target, state, histry,
    k_s, v_s,
    dropout_rate_s, dropout_enabled_s, rng_s, mask_s,
    w_dense, b_dense, num_l,
    mask_histry=None
):
    """Per-sample body. Operates on (N, b) slices."""
    # Prepare initial inputs
    q_ags_init = jnp.concatenate([actions, target, state], axis=0)
    seq_len, d_model = histry.shape[-2], histry.shape[-1]
    pe = get_sinusoidal_positional_encoding(seq_len, d_model)
    histry_pe_init = histry + pe

    def layer_fn(carry, _):
        q_ags, h_pe = carry

        # 1. actions, goal, state attention
        y_ags = nn.dot_product_attention(
            query=_add_head_axis(q_ags),
            key=_add_head_axis(k_s),
            value=_add_head_axis(v_s),
            bias=mask_s,
            dropout_rate=dropout_rate_s,
            deterministic=not dropout_enabled_s,
            dropout_rng=rng_s
        )
        y_ags = _drop_head_axis(y_ags)

        # 2. attention to history
        y_histry = nn.dot_product_attention(
            query=_add_head_axis(h_pe),
            key=_add_head_axis(h_pe),
            value=_add_head_axis(h_pe),
            bias=mask_histry,
            dropout_rate=dropout_rate_s,
            deterministic=not dropout_enabled_s,
            dropout_rng=rng_s
        )
        y_histry = _drop_head_axis(y_histry)

        # 3. linear integration and residual update
        y_combined = jnp.concatenate([y_ags, y_histry], axis=0)
        out = jnp.dot(y_combined, w_dense) + b_dense

        return (out[:q_ags.shape[0]], out[q_ags.shape[0]:]), None

    (final_q, final_h), _ = jax.lax.scan(layer_fn, (q_ags_init, histry_pe_init), jnp.arange(num_l))

    y_attention_state = jnp.concatenate([final_q, final_h], axis=0)
    return y_attention_state


@functools.partial(jax.jit, static_argnames=('num_l', 'dropout_enabled_s'))
def Y_attention_state_5_2(
    actions, target, state, histry,
    k_s, v_s,
    dropout_rate_s, dropout_enabled_s, rng_s, mask_s,
    w_dense, b_dense, num_l,
    mask_histry=None
):
    """
        idea5, type1 channel separation
        A function that separates the attention of actions,
        goals, and states from the self-attention of history, and integrates them using a fully connected layer.
        Layered according to num_l.

        Batch dimension added: actions, target, state, histry, k_s, v_s,
        mask_s, and mask_histry (when not None) now have shape (B, N, b).
        num_l and dropout_enabled_s are static (see 5_1_Bace docstring for
        why). dropout_rate_s, rng_s, w_dense, b_dense do not need to be
        static.
    """
    return jax.vmap(
        _Y_attention_state_5_2_single,
        in_axes=(0, 0, 0, 0, 0, 0, None, None, None, 0, None, None, None, 0)
    )(actions, target, state, histry, k_s, v_s, dropout_rate_s, dropout_enabled_s, rng_s, mask_s, w_dense, b_dense, num_l, mask_histry)

In [7]:
def _Y_attention_state_5_3_L_1_single(
    actions, target, state, histry,
    dropout_rate_s, dropout_enabled_s, rng_s,
    mask_q, mask_k, num_l, mask_histry=None
):
    """Per-sample body. Operates on (N, b) slices."""
    def make_qkv(actions, target, state, histry):
        q_seq_len = actions.shape[-2]
        q_d_model = actions.shape[-1]
        q_pe = get_sinusoidal_positional_encoding(q_seq_len, q_d_model)
        q = actions + q_pe

        k_bace = jnp.concatenate([target, state], axis=0)
        k_seq_len = k_bace.shape[-2]
        k_d_model = k_bace.shape[-1]
        k_pe = get_sinusoidal_positional_encoding(k_seq_len, k_d_model)
        k = k_bace + k_pe

        v_seq_len = histry.shape[-2]
        v_d_model = histry.shape[-1]
        v_pe = get_sinusoidal_positional_encoding(v_seq_len, v_d_model)
        v = histry + v_pe

        qkv = jnp.concatenate([q, k, v], axis=0)
        return (qkv, q_seq_len, k_seq_len, v_seq_len)


    def attention(carry, _):
        qkv = qkv_init + carry
        q, k, v = jnp.split(qkv, [q_split_idx, k_split_idx], axis=0)

        y_attention_state_carry_q = nn.dot_product_attention(
            query=_add_head_axis(q),
            key=_add_head_axis(q),
            value=_add_head_axis(q),
            bias=mask_q,
            dropout_rate=dropout_rate_s,
            deterministic=not dropout_enabled_s,
            dropout_rng=rng_s
        )
        y_attention_state_carry_k = nn.dot_product_attention(
            query=_add_head_axis(k),
            key=_add_head_axis(k),
            value=_add_head_axis(k),
            bias=mask_k,
            dropout_rate=dropout_rate_s,
            deterministic=not dropout_enabled_s,
            dropout_rng=rng_s
        )
        y_attention_state_carry_v = nn.dot_product_attention(
            query=_add_head_axis(v),
            key=_add_head_axis(v),
            value=_add_head_axis(v),
            bias=mask_histry,
            dropout_rate=dropout_rate_s,
            deterministic=not dropout_enabled_s,
            dropout_rng=rng_s
        )

        y_attention_state_carry = jnp.concatenate([
            _drop_head_axis(y_attention_state_carry_q),
            _drop_head_axis(y_attention_state_carry_k),
            _drop_head_axis(y_attention_state_carry_v)
        ], axis=0)

        return y_attention_state_carry, None

    qkv_init, q_seq_len, k_seq_len, v_seq_len = make_qkv(actions, target, state, histry)
    q_split_idx = q_seq_len
    k_split_idx = q_seq_len + k_seq_len

    carry_init = jnp.zeros_like(qkv_init)
    loop_trigger = jnp.arange(num_l)

    final_carry, _ = jax.lax.scan(attention, carry_init, loop_trigger)
    y_attention_state = qkv_init + final_carry

    return y_attention_state


@functools.partial(jax.jit, static_argnames=('num_l', 'dropout_enabled_s'))
def Y_attention_state_5_3_L_1(
    actions, target, state, histry,
    dropout_rate_s, dropout_enabled_s, rng_s,
    mask_q, mask_k, num_l, mask_histry=None
):
    """ idea5, type2 channel separation.
        channel is A T+S H (docstring previously said 'A T+S S'; the third
        channel is history, not a second state -- corrected label)
        Enable Position encoding

        Batch dimension added: actions, target, state, histry, mask_q,
        mask_k, and mask_histry (when not None) now have shape (B, N, b).
        num_l and dropout_enabled_s are static (see 5_1_Bace docstring).

        mask_s was previously a single shared bias for both the q branch
        (length = actions.shape[0]) and the k branch (length =
        target.shape[0] + state.shape[0]). Those lengths generally differ
        (10 vs 2 in testing), so one shared mask cannot be validly shaped
        for both attention calls; it is now two parameters, mask_q and
        mask_k, each shaped for its own branch.
    """
    return jax.vmap(
        _Y_attention_state_5_3_L_1_single,
        in_axes=(0, 0, 0, 0, None, None, None, 0, 0, None, 0)
    )(actions, target, state, histry, dropout_rate_s, dropout_enabled_s, rng_s, mask_q, mask_k, num_l, mask_histry)


In [8]:
def _Y_attention_state_5_4_single(
    actions, target, state, histry,
    mask_s,
    W_k, W_dense, b_dense,
    p_coe, r_coe, gamma_coe, k_mdp
):
    # 1. Self-attention to find relationships between 'now', 'goal', and 'actions'
    pe = get_sinusoidal_positional_encoding(histry.shape[0], histry.shape[1])
    histry_pe = histry + pe
    kv = jnp.concatenate([actions, target, histry_pe], axis=0)
    y_attn = nn.dot_product_attention(
        query=_add_head_axis(state),
        key=_add_head_axis(kv),
        value=_add_head_axis(kv),
        bias=mask_s
    )
    y_attn = _drop_head_axis(y_attn)

    # 2. Apply W_K with physical constraints (masking via softmax)
    # We assume W_k is a compatibility matrix
    constrained_features = jnp.dot(y_attn, W_k)

    # 3. Compress into small arrays to predict next state S'
    # Output shape targets (1, d)
    s_prime_pred = jnp.dot(constrained_features, W_dense) + b_dense
    s_prime = s_prime_pred[-1:, :] # Representative S'

    # 4. MDP Search for actions using Bellman equation
    # We construct the mdp_array using the predicted next state
    num_actions = actions.shape[0]
    state_rep = jnp.repeat(state, num_actions, axis=0)
    s_prime_rep = jnp.repeat(s_prime, num_actions, axis=0)

    # NOTE (placeholder, unresolved): mean-pooling each (num_actions, d)
    # embedding block down to one scalar per row makes the shapes line up
    # for _make_array_to_action_single, but pooling is not a validated way
    # to turn a continuous embedding into a discrete state/action id --
    # this is the same open design question flagged previously (how
    # attention output becomes an id for the tabular Bellman step). This
    # replaces the previous `.flatten()[:num_actions]`, which silently
    # grabbed arbitrary values from a repeated, flattened tensor with no
    # per-row correspondence to the actual action rows at all.
    mdp_array = _make_array_to_action_single(
        jnp.mean(state_rep, axis=-1),
        jnp.mean(s_prime_rep, axis=-1),
        jnp.mean(actions, axis=-1),
        p_coe, r_coe, gamma_coe
    )

    top_actions, top_values = _y_mdp_action_single(mdp_array, k_mdp)

    return top_actions, top_values, s_prime

@functools.partial(jax.jit, static_argnames=('k_mdp',))
def Y_attention_state_5_4(
    actions, target, state, histry,
    mask_s,
    W_k, W_dense, b_dense,
    p_coe, r_coe, gamma_coe, k_mdp
):
    """
    Batch dimension: actions, target, state, histry, mask_s have shape
    (B, N, b); W_k, W_dense, b_dense, p_coe, r_coe, gamma_coe are shared,
    unbatched. k_mdp is static (jax.lax.top_k requires a concrete value,
    same reasoning as y_mdp_action).

    Fixes relative to the draft this was built from:
    - in_axes had 13 entries for this function's 12 parameters (6 zeros +
      7 Nones); ValueError: vmap in_axes must be an int, None, or a tuple
      of entries corresponding to the positional arguments, confirmed.
      Corrected to 5 batched entries (actions, target, state, histry,
      mask_s) + 7 shared entries.
    - `mdp_array = y_mdp_action(state_rep..., ...)` called y_mdp_action
      (signature: (array_to_action, k)) with 6 positional arguments,
      which does not match its signature. The intent was to build the
      array, not solve it -- changed to
      `_make_array_to_action_single(...)`, and using the single-sample
      form (not the public vmapped make_array_to_action) to avoid a
      nested vmap, since this function is itself vmapped.
    """
    return jax.vmap(
        _Y_attention_state_5_4_single,
        in_axes=(0, 0, 0, 0, 0, None, None, None, None, None, None, None)
    )(actions, target, state, histry, mask_s, W_k, W_dense, b_dense, p_coe, r_coe, gamma_coe, k_mdp)

In [9]:
def _Get_next_state_single(y_attention_state, state_index):
        """Original per-sample body, unmodified."""
        state_softmax = nn.softmax(y_attention_state[state_index])
        state_next = jnp.argmax(state_softmax)
        return (state_next)


@jax.jit
def Get_next_state(y_attention_state, state_index):
        """ This is Transformer thought best next state.

        Batch dimension added: y_attention_state now has shape (B, N, b).
        state_index is treated as shared across the batch (the same
        relative position is queried for every sample); pass a batched
        index array with in_axes changed to 0 if per-sample indices are
        needed instead.
        """
        return jax.vmap(_Get_next_state_single, in_axes=(0, None))(y_attention_state, state_index)

In [10]:
def _y_attention_action_single(array_to_action, k_a, v_a, dropout_rate_a, dropout_enabled_a, rng_a, mask_a):
    """Per-sample body."""
    q = array_to_action

    y_attention_action = nn.dot_product_attention(
        query = _add_head_axis(q),
        key = _add_head_axis(k_a),
        value = _add_head_axis(v_a),
        bias = mask_a,
        dropout_rate = dropout_rate_a,
        deterministic = not dropout_enabled_a,
        dropout_rng = rng_a
    )

    return _drop_head_axis(y_attention_action)


# Next, i think think next action to allive to state_next. and to cut cost, maybe MDP is better. but this is study so use transformer.
@functools.partial(jax.jit, static_argnames=('dropout_enabled_a',))
def y_attention_action(array_to_action, k_a, v_a, dropout_rate_a, dropout_enabled_a, rng_a, mask_a):
    """This function return Next step **action**.
       but i think transformer is hevy. so just test
       array must include reword of action, % of go state when use actuion(i), gamma
       And, i think dropout must not use. It feels strange to train by logically dropping out groups of choices. I have no evidence for this.

       Batch dimension added: array_to_action, k_a, v_a, mask_a now have
       shape (B, N, b). dropout_enabled_a is static (same reasoning as
       5_1_Bace's dropout_enabled_s -- a Python `not` on it requires a
       concrete value under jit). dropout_rate_a, rng_a are shared,
       unbatched, and do not need to be static. Same missing-heads-axis
       fix applied here as in the other attention calls (untested by the
       existing test cell, but reproduced with the identical error in
       isolation).
    """
    return jax.vmap(
        _y_attention_action_single, in_axes=(0, 0, 0, None, None, None, 0)
    )(array_to_action, k_a, v_a, dropout_rate_a, dropout_enabled_a, rng_a, mask_a)

In [11]:
import traceback
import jax
import jax.numpy as jnp

# --- Test Configuration ---
B, N, D = 2, 10, 8
num_l = 2
num_step = 5
k_mdp = 3
p_coe, r_coe, gamma_coe = 0.1, 0.5, 0.9

# Prepare dummy inputs
key = jax.random.PRNGKey(20090424)
actions = jax.random.normal(key, (B, N, D))
target = jax.random.normal(key, (B, 1, D))
state = jax.random.normal(key, (B, 1, D))
histry = jax.random.normal(key, (B, 20, D))

# Shared weights/masks
k_s = jax.random.normal(key, (B, N + 22, D))
v_s = jax.random.normal(key, (B, N + 22, D))
mask_s = jnp.zeros((B, 1, N + 22, N + 22))
mask_h = jnp.zeros((B, 1, 20, 20))
w_dense = jax.random.normal(key, (D, D))
b_dense = jax.random.normal(key, (D,))

def raw_test_run(name, func, *args):
    print(f"\n{'='*20}\nTesting: {name}\n{'='*20}")
    try:
        res = func(*args)
        print(f"SUCCESS: {name}")
        if isinstance(res, tuple):
             print("Output shapes:", [getattr(r, 'shape', 'non-array') for r in res])
        else:
             print("Output shape:", res.shape)
    except Exception:
        print(f"FAILURE: {name}")
        traceback.print_exc()

# Execute Tests
raw_test_run("5_1 Bace", Y_attention_state_5_1_Bace,
             actions, target, state, histry, k_s, v_s, 0.0, False, key, mask_s, num_l, num_step, k_mdp)

# 5_2: q_ags length = actions(N) + target(1) + state(1) = N+2, not N+3 --
# the previous N+3 slice was one row too wide for k_s/v_s/mask_s here.
raw_test_run("5_2", Y_attention_state_5_2,
             actions, target, state, histry, k_s[:, :N+2, :], v_s[:, :N+2, :],
             0.0, False, key, mask_s[:, :, :N+2, :N+2], w_dense, b_dense, num_l, mask_h)

# 5_3: mask_s split into mask_q (q branch, length = actions = N) and
# mask_k (k branch, length = target + state = 2) -- see 5_3_L_1's
# docstring for why one shared mask could not work here.
mask_q_5_3 = jnp.zeros((B, 1, N, N))
mask_k_5_3 = jnp.zeros((B, 1, 2, 2))
raw_test_run("5_3 L_1", Y_attention_state_5_3_L_1,
             actions, target, state, histry, 0.0, False, key, mask_q_5_3, mask_k_5_3, num_l, mask_h)

# Testing 5_4
# state (query) length=1, kv length = actions(N) + target(1) + histry(20) = N+21
mask_5_4 = jnp.zeros((B, 1, 1, N + 21))
W_k = jax.random.normal(key, (D, D)) * 0.1
raw_test_run("5_4", Y_attention_state_5_4,
             actions, target, state, histry, mask_5_4,
             W_k, w_dense, b_dense, p_coe, r_coe, gamma_coe, k_mdp)

Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda12.initialize()
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/jax/_src/xla_bridge.py", line 508, in discover_pjrt_plugins
    plugin_module.initialize()
  File "/usr/local/lib/python3.11/dist-packages/jax_plugins/xla_cuda12/__init__.py", line 370, in initialize
    _check_cuda_versions(raise_on_first_error = True)
  File "/usr/local/lib/python3.11/dist-packages/jax_plugins/xla_cuda12/__init__.py", line 274, in _check_cuda_versions
    for d in range(cuda_versions.cuda_device_count())
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: jaxlib/cuda/versions_helpers.cc:135: operation cuInit(0) failed: Unknown CUDA error 303; cuGetErrorName failed. This probably means that JAX was unable to load the CUDA libraries.



Testing: 5_1 Bace
SUCCESS: 5_1 Bace
Output shapes: [(2, 10, 3), (2, 10, 3)]

Testing: 5_2
SUCCESS: 5_2
Output shape: (2, 32, 8)

Testing: 5_3 L_1
SUCCESS: 5_3 L_1
Output shape: (2, 32, 8)

Testing: 5_4
SUCCESS: 5_4
Output shapes: [(2, 10, 3), (2, 10, 3), (2, 1, 8)]
